1. Setup and Initialization

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize with a supported model name
chat_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # Updated from 1.5-flash
    temperature=0,
    api_key=userdata.get('GOOGLE_API_KEY')
)

messages = [
    SystemMessage(content="You're an assistant knowledgeable about healthcare. Only answer healthcare-related questions."),
    HumanMessage(content="What is Ayushman Bharat?"),
]

result = chat_model.invoke(messages)
print(result.content)

Ayushman Bharat is a flagship national healthcare initiative launched by the Government of India. Its primary aim is to achieve Universal Health Coverage (UHC) and ensure that no one is left behind when it comes to accessing quality healthcare services.

The scheme has two main components:

1.  **Pradhan Mantri Jan Arogya Yojana (PMJAY):** This is the world's largest government-funded health assurance scheme. It provides a health cover of up to INR 5 lakh (approximately USD 6,000) per family per year for secondary and tertiary care hospitalization to over 10.74 crore (107.4 million) poor and vulnerable families (approximately 50 crore or 500 million beneficiaries). It offers cashless and paperless access to services at empanelled public and private hospitals.

2.  **Ayushman Bharat Health and Wellness Centers (AB-HWCs):** These centers are designed to bring healthcare closer to the homes of people. They transform existing Primary Health Centres (PHCs) and Sub-Centres into AB-HWCs to pr

In [11]:
chat_model.invoke("What is blood pressure?")

AIMessage(content='Blood pressure is the **force of blood pushing against the walls of your arteries** as your heart pumps blood throughout your body.\n\nThink of it like the water pressure in a garden hose:\n*   The **pump** is your heart.\n*   The **water** is your blood.\n*   The **hose walls** are your artery walls.\n*   The **pressure** is how hard the water is pushing against those walls.\n\n**Why is it important?**\nThis pressure is essential to keep blood flowing, delivering oxygen and nutrients to all your organs and tissues. However, if the pressure is consistently too high (hypertension) or too low (hypotension), it can lead to serious health problems.\n\n**How is it measured?**\nBlood pressure is measured using a cuff (sphygmomanometer) and is expressed as two numbers, typically written as a fraction (e.g., 120/80 mmHg):\n\n1.  **Systolic Pressure (the top number):**\n    *   This is the **higher** number.\n    *   It measures the pressure in your arteries when your heart *

2. Defining Retrieval and Generation Logic

In [13]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Initialize the Chat Model with a supported model name
chat_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    api_key=userdata.get('GOOGLE_API_KEY')
)

# 2. Define the System Behavioral Context
system_message = SystemMessage(content="You're an assistant knowledgeable about healthcare. Only answer healthcare-related questions.")

# 3. List of 5 Healthcare Questions
questions = [
    "What is Ayushman Bharat?",
    "How can someone manage Type 2 Diabetes through diet?",
    "What are the common symptoms of a Vitamin D deficiency?",
    "What is the importance of vaccinations for infants?",
    "How does regular exercise improve cardiovascular health?"
]

# 4. Execute the questions
print("--- Healthcare Assistant Responses ---\n")

for i, q in enumerate(questions, 1):
    messages = [system_message, HumanMessage(content=q)]
    try:
        result = chat_model.invoke(messages)
        print(f"Question {i}: {q}")
        print(f"Answer: {result.content}\n")
        print("-" * 30)
    except Exception as e:
        print(f"Error on Question {i}: {e}")

# 5. The "Incorrect" (Out of Scope) Test Case
print("\n--- Out of Scope Test (Targeting 1 Incorrect Answer) ---")
out_of_scope_msg = [system_message, HumanMessage(content="How do I change a car tire?")]
refusal = chat_model.invoke(out_of_scope_msg)
print(f"Question: How do I change a car tire?")
print(f"Model Response: {refusal.content}")

--- Healthcare Assistant Responses ---

Question 1: What is Ayushman Bharat?
Answer: Ayushman Bharat is a flagship national health protection scheme launched by the Government of India. It aims to achieve Universal Health Coverage (UHC) and is designed to meet Sustainable Development Goals (SDGs) and "leave no one behind."

The scheme has two main components:

1.  **Pradhan Mantri Jan Arogya Yojana (PMJAY):** This is the world's largest government-funded health assurance scheme.
    *   **Objective:** To provide health cover to the poorest and most vulnerable families.
    *   **Coverage:** It provides a health cover of up to ₹5 lakh (approximately $6,000 USD) per family per year for secondary and tertiary care hospitalization.
    *   **Beneficiaries:** Over 10.74 crore (107.4 million) poor and vulnerable families (approximately 50 crore or 500 million individuals) based on the Socio-Economic Caste Census (SECC) 2011 data.
    *   **Nature:** It's a cashless and paperless scheme at pu

3. Creating the Gradio Chat Interface

In [ ]:
import gradio as gr
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# 1. Initialize the Gemini Model (Updated to a supported version)
chat_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    api_key=userdata.get('GOOGLE_API_KEY')
)

# 2. Local Knowledge Base (Since you have no AWS keys)
# This mimics the data your 'Retrieve' function would have found.
MOCK_KNOWLEDGE_BASE = {
    "ayushman bharat": "Ayushman Bharat is India's flagship health scheme providing ₹5 lakh insurance per family per year for secondary and tertiary care.",
    "diabetes": "Type 2 diabetes management involves a diet rich in whole grains, lean protein, and healthy fats while limiting processed sugars.",
    "vitamin d": "Common symptoms of Vitamin D deficiency include fatigue, bone pain, muscle weakness, and mood changes like depression.",
    "vaccination": "Infant vaccinations are crucial for building immunity against life-threatening diseases like polio, measles, and hepatitis B.",
    "exercise": "Regular exercise improves cardiovascular health by strengthening the heart muscle and improving blood circulation."
}

def respond_to_user_question(message, history):
    # Step 1: Mock Retrieval Logic
    query = message.lower()
    context = ""
    for key, value in MOCK_KNOWLEDGE_BASE.items():
        if key in query:
            context = value
            break

    # Step 2: Build the AI Prompt
    system_prompt = "You're an assistant knowledgeable about healthcare. Only answer healthcare-related questions."
    if context:
        system_prompt += f" Use this specific data to help answer: {context}"

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=message)
    ]

    # Step 3: Get Answer from Gemini
    try:
        result = chat_model.invoke(messages)
        return result.content
    except Exception as e:
        return f"Error: {e}"

# 3. Launch Gradio with the new 'messages' type to fix the warning
interface = gr.ChatInterface(
    fn=respond_to_user_question,
    type="messages", # This fixes the 'tuples' deprecation warning
    title="Healthcare Review Helper Bot"
)

interface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://262841124b3c163678.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
